In [166]:
import pandas as pd
import os
from os.path import dirname


root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/comuzzi/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/sebdis/ProcessMining/HGNN/HGNN_NA
/home/sebdis/ProcessMining/HGNN/HGNN_NA/data/datasets/comuzzi/
/home/sebdis/ProcessMining/HGNN/HGNN_NA/data/datasets/comuzzi/_processed/
/home/sebdis/ProcessMining/HGNN/HGNN_NA/data/datasets/graphs_repair/


In [167]:
#dataset = "small_log"
#dataset = "bpi_2013"
#dataset = "bpi_2012"
#dataset = "large_log"
#dataset = "sp2020"
dataset = "BPI20_RequestForPayment"


In [168]:
import json
with open("dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[f"{dataset}_CZ"]
dataset_info

{'categorical': ['org:resource',
  'Activity',
  'org:role',
  'case:Project',
  'case:Task',
  'case:OrganizationalEntity',
  'case:Activity',
  'case:RfpNumber'],
 'numerical': ['time:timestamp', 'case:RequestedAmount']}

In [169]:
nan_methods = ["odd", "even", "window", "random", "attr_level"]

In [170]:
raw_data = pd.read_csv(f"{data_dir}/{dataset}/complete_df_full_even.csv")
raw_data.head()

,CaseID,Activity,CompleteTimestamp,org:resource,org:role,case:Project,case:Task,case:OrganizationalEntity,case:Cost Type,case:RequestedAmount,case:Activity,case:RfpNumber,CumTimeInterval
0,1,Request For Payment SUBMITTED by EMPLOYEE,2017-01-09 09:17:18+00:00,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215,0.0
1,1,Request For Payment FINAL_APPROVED by SUPERVISOR,2017-01-09 09:18:00+00:00,STAFF MEMBER,SUPERVISOR,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215,42.0
2,1,Request For Payment REJECTED by MISSING,2017-01-10 12:42:32+00:00,STAFF MEMBER,MISSING,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215,98714.0
3,1,Request For Payment SUBMITTED by EMPLOYEE,2017-03-03 09:51:13+00:00,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215,4581235.0
4,1,Request For Payment APPROVED by PRE_APPROVER,2017-03-03 09:51:42+00:00,STAFF MEMBER,PRE_APPROVER,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215,4581264.0


In [171]:
if dataset == "sp2020":
    raw_data.fillna({"org:resource": "EMPTY"})

In [172]:
raw_data.columns

Index(['CaseID', 'Activity', 'CompleteTimestamp', 'org:resource', 'org:role',
       'case:Project', 'case:Task', 'case:OrganizationalEntity',
       'case:Cost Type', 'case:RequestedAmount', 'case:Activity',
       'case:RfpNumber', 'CumTimeInterval'],
      dtype='object')

In [173]:
if dataset == "bpi_2013" or dataset == "small_log" or dataset == "large_log" or dataset == "sp2020":
    date_format = '%Y-%m-%d %H:%M:%S'
elif dataset == "bpi_2012":
    date_format = '%Y-%m-%d %H:%M:%S.%f'
elif dataset == "BPI20_RequestForPayment":
    date_format = "%Y-%m-%d %H:%M:%S%z"
else:
    date_format = '%Y/%m/%d %H:%M:%S.%f'
date_format

'%Y-%m-%d %H:%M:%S%z'

In [174]:
from datetime import datetime

def translate_time(time_str):
    return datetime.strptime(time_str, date_format).timestamp()

In [175]:
train_dataset = pd.read_csv(f"{data_dir}/{dataset}/complete_df_train_even.csv")
valid_dataset = pd.read_csv(f"{data_dir}/{dataset}/complete_df_val_even.csv")
test_dataset = pd.read_csv(f"{data_dir}/{dataset}/complete_df_test_even.csv")

In [176]:
masked_datasets = {key : pd.read_csv(f"{data_dir}/{dataset}/missing_df_full_{key}.csv") for key in nan_methods}

In [177]:
def restore_missing_classes(x,y):
    class_x = set(x)
    for i in range(len(x)):
        if (y[i] not in class_x):
           x[i] = y[i]
    return x 

In [178]:
def restore_dataset(data:pd.DataFrame, original_data:pd.DataFrame):
    columns = dataset_info["categorical"]
    for k in columns:
        data[k] = restore_missing_classes(data[k].values, original_data[k].values)
    return data

In [179]:
if dataset == "bpi_2013" or dataset == "bpi_2012":
    tab_all = raw_data.rename(columns={"CompleteTimestamp": "time:timestamp", "Resource": "org:resource"})
    train_dataset = train_dataset.rename(columns={"CompleteTimestamp": "time:timestamp", "Resource": "org:resource"})
    valid_dataset = valid_dataset.rename(columns={"CompleteTimestamp": "time:timestamp", "Resource": "org:resource"})
    test_dataset = test_dataset.rename(columns={"CompleteTimestamp": "time:timestamp", "Resource": "org:resource"})
    for k in masked_datasets:
        masked_datasets[k] = masked_datasets[k].rename(columns={"CompleteTimestamp": "time:timestamp", "Resource": "org:resource"})
elif dataset == "small_log" or dataset == "large_log" or dataset == "sp2020" or dataset == "BPI20_RequestForPayment":
    tab_all = raw_data.rename(columns={"CompleteTimestamp": "time:timestamp"})
    train_dataset = train_dataset.rename(columns={"CompleteTimestamp": "time:timestamp"})
    valid_dataset = valid_dataset.rename(columns={"CompleteTimestamp": "time:timestamp"})
    test_dataset = test_dataset.rename(columns={"CompleteTimestamp": "time:timestamp"})
    for k in masked_datasets:
        masked_datasets[k] = masked_datasets[k].rename(columns={"CompleteTimestamp": "time:timestamp"})


In [180]:
for k in masked_datasets:
    masked_datasets[k] = restore_dataset(masked_datasets[k], tab_all)

In [181]:
tab_all["time:timestamp"] = tab_all["time:timestamp"].apply(translate_time)
train_dataset["time:timestamp"] = train_dataset["time:timestamp"].apply(translate_time)
valid_dataset["time:timestamp"] = valid_dataset["time:timestamp"].apply(translate_time)
test_dataset["time:timestamp"] = test_dataset["time:timestamp"].apply(translate_time)

for k in masked_datasets:
    masked_datasets[k]["time:timestamp"] = [translate_time(x) if type(x) == str else x for x in  masked_datasets[k]["time:timestamp"].values]

In [182]:
from math import log

def log_norm(x):
    return log(x+1) 
 
from numpy import NaN 

#if dataset == "bpi_2012":
#    tab_all["(case) AMOUNT_REQ"] = [log(x) if x > 0 else x if x is NaN else 0. for x in  tab_all["(case) AMOUNT_REQ"].values]
#    train_dataset["(case) AMOUNT_REQ"] = [log(x) if x > 0 else x if x is NaN else 0. for x in  train_dataset["(case) AMOUNT_REQ"].values]
#    valid_dataset["(case) AMOUNT_REQ"] = [log(x) if x > 0 else x if x is NaN else 0. for x in  valid_dataset["(case) AMOUNT_REQ"].values]
#    test_dataset["(case) AMOUNT_REQ"] = [log(x) if x > 0 else x if x is NaN else 0. for x in  test_dataset["(case) AMOUNT_REQ"].values]
#
#    for k in masked_datasets:
#        masked_datasets[k]["(case) AMOUNT_REQ"] = [log(x) if x > 0 else x if x is NaN else 0. for x in  masked_datasets[k]["(case) AMOUNT_REQ"].values]

if dataset == "bpi_2012":
    tab_all["(case) AMOUNT_REQ"] = [ x if x is NaN else log_norm(x) for x in  tab_all["(case) AMOUNT_REQ"].values]
    train_dataset["(case) AMOUNT_REQ"] = [x if x is NaN else log_norm(x) for x in  train_dataset["(case) AMOUNT_REQ"].values]
    valid_dataset["(case) AMOUNT_REQ"] = [x if x is NaN else log_norm(x) for x in  valid_dataset["(case) AMOUNT_REQ"].values]
    test_dataset["(case) AMOUNT_REQ"] = [x if x is NaN else log_norm(x) for x in  test_dataset["(case) AMOUNT_REQ"].values]

    for k in masked_datasets:
        masked_datasets[k]["(case) AMOUNT_REQ"] = [x if x is NaN else log_norm(x) for x in  masked_datasets[k]["(case) AMOUNT_REQ"].values]

In [183]:
tab_all = tab_all.drop(columns=["CumTimeInterval"])
train_dataset = train_dataset.drop(columns=["CumTimeInterval"])
valid_dataset = valid_dataset.drop(columns=["CumTimeInterval"])
test_dataset = test_dataset.drop(columns=["CumTimeInterval"])

for k in masked_datasets:
    masked_datasets[k] = masked_datasets[k].drop(columns=["CumTimeInterval"])

In [184]:
min = tab_all["time:timestamp"].min()
min

1483953438.0

In [185]:
from numpy import NaN


tab_all["time:timestamp"] -= min
train_dataset["time:timestamp"] -= min
valid_dataset["time:timestamp"] -= min 
test_dataset["time:timestamp"] -= min 

for k in masked_datasets:
    masked_datasets[k]["time:timestamp"] = [x-min if x is not NaN else x for x in  masked_datasets[k]["time:timestamp"].values]

In [186]:
from math import log

#tab_all["time:timestamp"] = [log(x) if x > 0 else 0. for x in tab_all["time:timestamp"].values ]
#train_dataset["time:timestamp"] = [log(x) if x > 0 else 0. for x in train_dataset["time:timestamp"].values ]
#valid_dataset["time:timestamp"] = [log(x) if x > 0 else 0. for x in valid_dataset["time:timestamp"].values ]
#test_dataset["time:timestamp"] = [log(x) if x > 0 else 0. for x in test_dataset["time:timestamp"].values ]
#
#for k in masked_datasets:
#    masked_datasets[k]["time:timestamp"] = [log(x) if x is not NaN and x != 0 else 0. for x in  masked_datasets[k]["time:timestamp"].values]
tab_all["time:timestamp"] = tab_all["time:timestamp"].apply(log_norm)
train_dataset["time:timestamp"] = train_dataset["time:timestamp"].apply(log_norm)
valid_dataset["time:timestamp"] = valid_dataset["time:timestamp"].apply(log_norm)
test_dataset["time:timestamp"] = test_dataset["time:timestamp"].apply(log_norm)

for k in masked_datasets:
    masked_datasets[k]["time:timestamp"] = [log(x+1) if x is not NaN  else 0. for x in  masked_datasets[k]["time:timestamp"].values]

In [187]:
tab_all.head()

,CaseID,Activity,time:timestamp,org:resource,org:role,case:Project,case:Task,case:OrganizationalEntity,case:Cost Type,case:RequestedAmount,case:Activity,case:RfpNumber
0,1,Request For Payment SUBMITTED by EMPLOYEE,0.000000,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
1,1,Request For Payment FINAL_APPROVED by SUPERVISOR,3.761200,STAFF MEMBER,SUPERVISOR,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
2,1,Request For Payment REJECTED by MISSING,11.499992,STAFF MEMBER,MISSING,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
3,1,Request For Payment SUBMITTED by EMPLOYEE,15.337479,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
4,1,Request For Payment APPROVED by PRE_APPROVER,15.337486,STAFF MEMBER,PRE_APPROVER,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215


In [188]:
train_dataset.head()

,CaseID,Activity,time:timestamp,org:resource,org:role,case:Project,case:Task,case:OrganizationalEntity,case:Cost Type,case:RequestedAmount,case:Activity,case:RfpNumber
0,1,Request For Payment SUBMITTED by EMPLOYEE,0.000000,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
1,1,Request For Payment FINAL_APPROVED by SUPERVISOR,3.761200,STAFF MEMBER,SUPERVISOR,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
2,1,Request For Payment REJECTED by MISSING,11.499992,STAFF MEMBER,MISSING,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
3,1,Request For Payment SUBMITTED by EMPLOYEE,15.337479,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
4,1,Request For Payment APPROVED by PRE_APPROVER,15.337486,STAFF MEMBER,PRE_APPROVER,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215


In [189]:
valid_dataset.head()

,CaseID,Activity,time:timestamp,org:resource,org:role,case:Project,case:Task,case:OrganizationalEntity,case:Cost Type,case:RequestedAmount,case:Activity,case:RfpNumber
0,4132,Request For Payment SUBMITTED by EMPLOYEE,17.671616,STAFF MEMBER,EMPLOYEE,project 147649,UNKNOWN,organizational unit 65463,0,371.320374,UNKNOWN,request for payment number 160559
1,4132,Request For Payment APPROVED by ADMINISTRATION,17.671616,STAFF MEMBER,ADMINISTRATION,project 147649,UNKNOWN,organizational unit 65463,0,371.320374,UNKNOWN,request for payment number 160559
2,4132,Request For Payment FINAL_APPROVED by SUPERVISOR,17.673380,STAFF MEMBER,SUPERVISOR,project 147649,UNKNOWN,organizational unit 65463,0,371.320374,UNKNOWN,request for payment number 160559
3,4132,Request Payment,17.675241,SYSTEM,UNDEFINED,project 147649,UNKNOWN,organizational unit 65463,0,371.320374,UNKNOWN,request for payment number 160559
4,4132,Payment Handled,17.682721,SYSTEM,UNDEFINED,project 147649,UNKNOWN,organizational unit 65463,0,371.320374,UNKNOWN,request for payment number 160559


In [190]:
test_dataset.head()

,CaseID,Activity,time:timestamp,org:resource,org:role,case:Project,case:Task,case:OrganizationalEntity,case:Cost Type,case:RequestedAmount,case:Activity,case:RfpNumber
0,5509,Request For Payment SAVED by EMPLOYEE,17.840750,STAFF MEMBER,EMPLOYEE,UNKNOWN,UNKNOWN,organizational unit 65468,0,40.304147,UNKNOWN,UNKNOWN
1,5510,Request For Payment SUBMITTED by EMPLOYEE,17.840769,STAFF MEMBER,EMPLOYEE,project 152803,task 176154,organizational unit 65457,0,18.087498,activity 505,request for payment number 176153
2,5510,Request For Payment APPROVED by ADMINISTRATION,17.840769,STAFF MEMBER,ADMINISTRATION,project 152803,task 176154,organizational unit 65457,0,18.087498,activity 505,request for payment number 176153
3,5510,Request For Payment FINAL_APPROVED by SUPERVISOR,17.848379,STAFF MEMBER,SUPERVISOR,project 152803,task 176154,organizational unit 65457,0,18.087498,activity 505,request for payment number 176153
4,5510,Request Payment,17.849001,SYSTEM,UNDEFINED,project 152803,task 176154,organizational unit 65457,0,18.087498,activity 505,request for payment number 176153


In [191]:
dataset = f"{dataset}_CZ"
dataset

'BPI20_RequestForPayment_CZ'

In [192]:
data_dir_processed

'/home/sebdis/ProcessMining/HGNN/HGNN_NA/data/datasets/comuzzi/_processed/'

In [193]:
if not os.path.isdir(f"{data_dir_processed}/{dataset}"):
    os.mkdir(f"{data_dir_processed}/{dataset}")

data_dir_processed = f"{data_dir_processed}/{dataset}/"

In [194]:
tab_all.to_csv(data_dir_processed + f"{dataset}_processed_all.csv", index=False)

In [195]:
train_dataset.to_csv(data_dir_processed+ f"{dataset}_processed_train.csv", index = False)

In [196]:
valid_dataset.to_csv(data_dir_processed+f"{dataset}_processed_valid.csv", index = False)

In [197]:
test_dataset.to_csv(data_dir_processed+ f"{dataset}_processed_test.csv", index = False)

In [198]:
for k in masked_datasets:
    masked_datasets[k].to_csv(data_dir_processed + f"{dataset}_masked_{k}_all.csv", index=False)